# Prompt evaluation

Requirements:
- Use preferrably Python `3.12.13` venvs.
    - Using older version might not stream events chunk by chunk, instead it might wait for the **entire** response before printing it.
- Add a `.env` file in the same folder as this notebook with a fields:
    - ANTHROPIC_BASE_URL = `<url_here>`
    - ANTHROPIC_AUTH_TOKEN = "`<jwt_token_here>`"

## ATENTION

- Prefer "claude-4-5-haiku" for this notebook.

## 0. Setup

### 0.1. Env and Client

In [1]:
# Load env variables and create client

## 1 Env
from dotenv import load_dotenv

load_dotenv()

## 2 Client
from anthropic import Anthropic

client = Anthropic()
model = "bedrock/anthropic.claude-4-5-haiku"

### 0.2. Helper functions

In [2]:
# Helper functions
def add_user_message(
    messages, 
    text,
):
    user_message = {
        "role": "user", 
        "content": text,
    }
    
    messages.append(user_message)


def add_assistant_message(
    messages, 
    text,
):
    assistant_message = {
        "role": "assistant", 
        "content": text,
    }

    messages.append(assistant_message)


def chat(
    messages, 
    system=None, 
    temperature=1.0, 
    stop_sequences=[],
):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system
    
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message.content[0].text

## 1. Prompt Eval

In [3]:
## Dataset generation helper

import json

def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []

    add_user_message(
        messages, 
        prompt,
    )

    prefill = "```json"
    add_assistant_message(
        messages,
        prefill,
    )

    stop_sequences = ["```"]
    text = chat(
        messages,
        stop_sequences=stop_sequences,
    )

    return json.loads(text)

In [4]:
## Generate dataset and print it

# dataset = generate_dataset()
# print(dataset)

In [5]:
## Save dataset to a file

# with open('dataset.json', 'w') as f:
#     json.dump(
#         dataset, 
#         f, 
#         indent=2,
#     )

# print("Dataset saved to dataset.json")

## 2. Running the eval

In [6]:
# Helpers

def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [7]:
# Read and run eval pipeline

with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [8]:
# Print results

print(json.dumps(results, indent=1))

[
 {
  "output": "# Extract AWS Region from S3 Bucket URL\n\nHere are several solutions depending on your needs:\n\n## 1. **Regex Solution (Most Reliable)**\n\n```python\nimport re\n\ndef extract_aws_region(url):\n    \"\"\"Extract AWS region from S3 bucket URL\"\"\"\n    match = re.search(r's3[.-]([a-z0-9\\-]+)\\.amazonaws\\.com', url)\n    if match:\n        return match.group(1)\n    return None\n\n# Examples\nprint(extract_aws_region('s3://my-bucket.s3.us-west-2.amazonaws.com/key'))\n# Output: us-west-2\n\nprint(extract_aws_region('s3://my-bucket.s3.amazonaws.com/key'))\n# Output: None (default region, not specified)\n\nprint(extract_aws_region('https://my-bucket.s3.eu-central-1.amazonaws.com/key'))\n# Output: eu-central-1\n```\n\n## 2. **URL Parsing Solution**\n\n```python\nfrom urllib.parse import urlparse\n\ndef extract_aws_region(url):\n    \"\"\"Extract AWS region using URL parsing\"\"\"\n    parsed = urlparse(url)\n    netloc = parsed.netloc\n    \n    # Pattern: bucket.s3.re

## 3. Model based grader

### 3.1. Helpers

In [9]:
# New grader function

def grade_by_model(
    test_case,
    output,
): 
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []

    prefill_sequence = "```json"
    stop_sequence = "```"

    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, prefill_sequence)

    eval_text = chat(
        messages,
        temperature=0,
        stop_sequences=[stop_sequence],
    )

    return json.loads(eval_text)

In [10]:
# Eval helpers override

from statistics import mean

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # Grader
    model_grade = grade_by_model(
        test_case,
        output,
    )

    score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    strengths = model_grade["strengths"]
    weaknesses = model_grade["weaknesses"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
        "strengths": strengths,
        "weaknesses": weaknesses,
    }

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    # Average grader score
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score:.2f}")

    return results

### 3.2. Read dataset

In [11]:
# Open dataset.json

with open("dataset.json", "r") as f:
    dataset = json.load(f)

### 3.3. Run eval

In [13]:
# Run eval pipeline

results = run_eval(dataset)

Average score: 7.00


In [14]:
# Print results

print(json.dumps(results, indent=2))

[
  {
    "output": "# Extract AWS Region from S3 Bucket URL\n\nHere are several solutions depending on your needs:\n\n## Solution 1: Using Regex (Most Robust)\n```python\nimport re\n\ndef extract_region_from_s3_url(url):\n    \"\"\"Extract AWS region from S3 bucket URL\"\"\"\n    # Pattern matches: s3.REGION.amazonaws.com\n    match = re.search(r's3[.-]([a-z0-9-]+)\\.amazonaws\\.com', url)\n    if match:\n        return match.group(1)\n    return None\n\n# Test cases\nprint(extract_region_from_s3_url('s3://my-bucket.s3.us-west-2.amazonaws.com/key'))\n# Output: us-west-2\n\nprint(extract_region_from_s3_url('s3://my-bucket.s3.eu-central-1.amazonaws.com/key'))\n# Output: eu-central-1\n\nprint(extract_region_from_s3_url('https://my-bucket.s3.ap-southeast-1.amazonaws.com/key'))\n# Output: ap-southeast-1\n```\n\n## Solution 2: Using String Splitting\n```python\ndef extract_region_from_s3_url(url):\n    \"\"\"Extract AWS region using string operations\"\"\"\n    if 'amazonaws.com' not in url

## 4. Code grader

### 4.1 New dataset with "format" field

In [15]:
# Generate dataset override - now supports format field

import json

def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []

    add_user_message(
        messages, 
        prompt,
    )

    prefill = "```json"
    add_assistant_message(
        messages,
        prefill,
    )

    stop_sequences = ["```"]
    text = chat(
        messages,
        stop_sequences=stop_sequences,
    )

    return json.loads(text)

In [16]:
# Generate the dataset and write it to 'dataset.json'

dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

### 4.2. Code graders

In [17]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


### 4.3 Helper overrides

In [18]:
# Run prompt override - now supports "format" field

def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""
    
    messages = []

    add_user_message(messages, prompt)

    prefill = "```code"
    add_assistant_message(messages, prefill)

    stop_sequence = "```"
    output = chat(messages, stop_sequences=[stop_sequence])
    
    return output

In [19]:
# Run test case override - now grades both model and syntax

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # Model grader
    model_grade = grade_by_model(
        test_case,
        output,
    )

    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    strengths = model_grade["strengths"]
    weaknesses = model_grade["weaknesses"]

    # Code grader
    syntax_score = grade_syntax(
        output, 
        test_case,
    )

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
        "strengths": strengths,
        "weaknesses": weaknesses,
    }

### 4.4 Run complete eval

In [20]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 8.17


In [21]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport re\nimport sys\n\ndef extract_s3_bucket(s3_uri):\n    match = re.match(r's3://([a-z0-9.-]+)(/|$)', s3_uri)\n    if match:\n        return match.group(1)\n    return None\n\nif __name__ == \"__main__\":\n    s3_uri = sys.argv[1] if len(sys.argv) > 1 else \"s3://my-bucket/path/to/file.txt\"\n    print(extract_s3_bucket(s3_uri))\n",
    "test_case": {
      "task": "Parse an AWS S3 bucket name from a full S3 URI (e.g., 's3://my-bucket/path/to/file.txt') and extract just the bucket name",
      "format": "regex"
    },
    "score": 8.5,
    "reasoning": "The solution correctly solves the core task with a working regex pattern that extracts bucket names from standard S3 URIs. However, it has minor robustness issues: the lowercase-only character class is overly restrictive for input validation (users might pass uppercase), and there's no error messaging when parsing fails. The code is functional but lacks production-ready error handling and documentation.",
    